Project 2: Trees, Rules, and the AI Audit (Person A)

Task 1 preprocessing, decision tree tuning (Task 3), and the RIPPER rule-based model.

Shared functions live in common.py and the RIPPER wrapper lives in rules.py. See PersonA_Code_Explainer.md for what each block does and why.

In [ ]:
# this is for RIPPER
%pip install wittgenstein

In [ ]:
# autoreload picks up edits to common.py / rules.py without restarting the kernel
%load_ext autoreload
%autoreload 2

import os
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

from common import *
from rules import RipperClassifier, make_rule_preprocessor, make_ripper_pipeline

Task 1

In [ ]:
#load the data and check it matches the spec
X, y = load_data()

print("Rows:", len(X))                      # expect 32561
print("Share >50K:", round(y.mean(), 3))    # expect ~0.241

In [ ]:
# checking to see what has missing values
X.isna().sum()

In [ ]:
# inspecting the shape 
print(make_preprocessor().fit_transform(X).shape)

Task 2

In [ ]:
# we collect every model's results here
results = {}

# small helper so every model prints the same way mean ± std for each metric
def show(name, res):
    print(name)
    for m, (mean, std) in res["metrics"].items():
        print(f"  {m:<10} {mean:.3f} ± {std:.3f}")

In [ ]:
#baseline pipeline 
dummy_pipe = Pipeline(
    steps=[
        ("prep", make_preprocessor()),
        ("clf", DummyClassifier(strategy="most_frequent"))])

results["Dummy (majority)"] = evaluate(dummy_pipe, X, y)

show("Dummy (majority)", results["Dummy (majority)"])

In [ ]:
#default tree
dt_default = Pipeline(
    steps=[
        ("prep", make_preprocessor()),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))])

results["Decision Tree (default)"] = evaluate(dt_default, X, y)

show("Decision Tree (default)", results["Decision Tree (default)"])

Task 3 

In [ ]:
dt_pipe = Pipeline(
    steps=[
        ("prep", make_preprocessor()),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))])

param_grid = {
    "clf__max_depth": list(range(3, 21)),                  
    "clf__min_samples_split": [2, 10, 50, 100, 200],       
    "clf__criterion": ["gini", "entropy"],
}

print("Candidates:", 18 * 5 * 2, "x 5 folds =", 18 * 5 * 2 * 5, "fits")

In [ ]:
#winner by F1 
dt_search = GridSearchCV(
    dt_pipe,
    param_grid=param_grid,
    scoring=SCORING,
    refit="f1",
    cv=CV,
    n_jobs=-1,
    return_train_score=True,
)
dt_search.fit(X, y)  

print("Best parameters:", dt_search.best_params_)
print("Best mean CV F1:", round(dt_search.best_score_, 4))

In [ ]:
#top 10 candidates by F1, with the other metrics alongside
cv_table = pd.DataFrame(dt_search.cv_results_)
cols = ["param_clf__criterion", "param_clf__max_depth", "param_clf__min_samples_split",
        "mean_test_f1", "std_test_f1", "mean_train_f1", "mean_test_roc_auc", "mean_test_accuracy"]
cv_table.sort_values("rank_test_f1")[cols].head(10).round(4)

In [ ]:
best_split = dt_search.best_params_["clf__min_samples_split"]
fig, ax = plt.subplots(figsize=(9, 5))

for criterion, color in [("gini", "tab:blue"), ("entropy", "tab:orange")]:
    rows = cv_table[(cv_table["param_clf__criterion"] == criterion) &
                    (cv_table["param_clf__min_samples_split"] == best_split)].sort_values("param_clf__max_depth")
    depth = rows["param_clf__max_depth"].astype(int)
    #test folds with a ±1 std band, training folds dashed
    ax.plot(depth, rows["mean_test_f1"], "-", color=color, label=f"{criterion} (test)")
    ax.fill_between(depth, rows["mean_test_f1"] - rows["std_test_f1"],
                    rows["mean_test_f1"] + rows["std_test_f1"], color=color, alpha=0.15)
    ax.plot(depth, rows["mean_train_f1"], "--", color=color, alpha=0.7, label=f"{criterion} (train)")

best_depth = dt_search.best_params_["clf__max_depth"]
ax.axvline(best_depth, linestyle=":", color="gray")
ax.annotate(f"best: depth {best_depth}", (best_depth + 0.2, dt_search.best_score_ - 0.03))
ax.set_xticks(range(3, 21))
ax.set_xlabel("max_depth")
ax.set_ylabel("F1 (positive class >50K)")
ax.set_title(f"Decision tree F1 vs. depth (min_samples_split={best_split})")
ax.legend()
plt.show()

In [ ]:
#evaluate() clones the pipeline, so this re-fits the best settings fold by fold
dt_best = dt_search.best_estimator_
results["Decision Tree (tuned)"] = evaluate(dt_best, X, y)
show("Decision Tree (tuned)", results["Decision Tree (tuned)"])

RIPPER 

In [ ]:
ripper_pipe = make_ripper_pipeline()

results["RIPPER"] = evaluate(ripper_pipe, X, y)
show("RIPPER", results["RIPPER"])

In [ ]:
ripper_full = make_ripper_pipeline().fit(X, y)
rules = ripper_full.named_steps["clf"].ruleset_

print("Number of rules:", len(rules))
print("Each rule predicts >50K; anything no rule covers is predicted <=50K\n")
for rule in rules[:10]:
    print(" ", rule)

In [ ]:
#one row per model, mean ± std from the same 5 folds
summary = pd.DataFrame({
    name: {m: f"{mean:.3f} ± {std:.3f}" for m, (mean, std) in res["metrics"].items()}
    for name, res in results.items()
}).T
summary

In [ ]:
#we save the results (including out-of-fold probabilities) so Task 4 and the final notebook can reuse them
os.makedirs("artifacts", exist_ok=True)
joblib.dump({"results": results, "dt_best_params": dt_search.best_params_},
            "artifacts/a_results.joblib")
print("saved artifacts/a_results.joblib")